In [ ]:
import numpy as np
import glob
import os
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})
colorVec = [
    "#910000",
    "#001AFF",
    '#FF5733',
    "#119100",
    "#B700FF",
]

In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=2):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
        vec = vec[~np.isnan(vec)]
    f.close()
    return vec

def read_col_csv(filePath, colIdx, skipHeader):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx, delimiter=',')
    f.close()
    return vec

def get_probe_coord_value(filePath, coordIdx):
    with open(filePath, 'r') as f:
        header = f.readline()
        coordValue = float(header.split('(')[1].split(')')[0].split(',')[coordIdx])
    return coordValue

def calculate_average(lists):
    return [sum(values) / len(lists) for values in zip(*lists)]

In [ ]:
x = 1.06
U0 = 0.039
avgRg = [20000, 60000]

folderVec = [
    f"/Users/maggie/repo/scripts/cases/flow_past_cylinder/3900_cml_smag017_40dx_intpbb/probes"
]
print(folderVec)

labelVec = [
    'LBM Cumulant'
]

yVec = np.linspace(-1, 1, 5)
zVec = np.linspace(-1.5, 1.5, 101)


In [ ]:


fileListVec = []
uxVec = []

for case in range(len(folderVec)):
    # for y in yVec:
    spaceAvgUxVec = [] # size: len(folderVec)
    spaceAvgUzVec = [] # size: len(folderVec)
    timeAvgUxVecVec = [] # size: len(yVec)
    timeAvgUzVecVec = [] # size: len(yVec)
    for iy in range(0, len(yVec)):
        yStr = "{:.1f}".format(yVec[iy])
        path = f"{folderVec[case]}/probeX{str(x)}Y{yStr}"
        pattern = os.path.join(f"{path}", "probe*.txt")
        fileList = glob.glob(pattern)
        fileList.sort()
        #! calculate time averaged U for one Y coord
        timeAvgUxVec = [] # size: len(zVec)
        timeAvgUzVec = [] # size: len(zVec)
        for loc in range(len(fileList)):
            file = fileList[loc]
            step = read_col_probe(file, 0, 2)
            intervalStep = step[1]-step[0]
            idxStart = int(np.abs(step - avgRg[0]).argmin())
            idxEnd = int(np.abs(step - avgRg[1]).argmin())
            ux = read_col_probe(file, 3, 2)
            uz = read_col_probe(file, 4, 2)
            timeAvgUx = np.mean(ux[idxStart:idxEnd]) / U0
            timeAvgUz = np.mean(uz[idxStart:idxEnd]) / U0
            timeAvgUxVec.append(timeAvgUx)
            timeAvgUzVec.append(timeAvgUz)

        timeAvgUxVecVec.append(timeAvgUxVec)
        timeAvgUzVecVec.append(timeAvgUzVec)
    #! calculate space averaged U
    spaceAvgUxVec.append(calculate_average(timeAvgUxVecVec))
    spaceAvgUzVec.append(calculate_average(timeAvgUzVecVec))

refUPath = f"../ref/Jacob-x{str(x)}-u.csv"
refVPath = f"../ref/Jacob-x{str(x)}-v.csv"
refUX = read_col_csv(refUPath, 0, 1)
refUU = read_col_csv(refUPath, 1, 1)
refVX = read_col_csv(refVPath, 0, 1)
refVV = read_col_csv(refVPath, 1, 1)


In [ ]:
figUx, axUx = plt.subplots(1, 1, figsize=(8, 4), facecolor='w', edgecolor='w')
figUz, axUz = plt.subplots(1, 1, figsize=(8, 4), facecolor='w', edgecolor='w')

if x == 1.06:
    # uxYlim = [-0.25, 1.5]
    uxYlim = [-0.5, 1.5]
    # uzYlim = [-0.15, 0.15]
    uzYlim = [-0.2, 0.2]
elif x == 1.54:
    uxYlim = [-0.5, 1.5]
    uzYlim = [-0.4, 0.4]
elif x == 2.02:
    uxYlim = [-0.4, 1.4]
    uzYlim = [-0.4, 0.4]

axUx.plot(refUX, refUU, lw=0, marker='o', ms=6, fillstyle='none', label='Parnaudeau et al., PIV', c='k')
axUz.plot(refVX, refVV, lw=0, marker='o', ms=6, fillstyle='none', label='Parnaudeau et al., PIV', c='k')

for case in range(len(folderVec)):
    axUx.plot(zVec, spaceAvgUxVec[case], label=labelVec[case], c=colorVec[case%len(folderVec)])
    axUx.set_xlabel(r"z/D")
    axUx.set_ylabel(r"$<u>/U_0$")
    axUx.set_xlim([-1.5, 1.5])
    axUx.set_ylim(uxYlim)
    axUx.legend(loc='upper right', frameon=False, bbox_to_anchor=(0.8, -0.15), ncol=2)


    axUz.plot(zVec, spaceAvgUzVec[case], label=labelVec[case], c=colorVec[case%len(folderVec)])
    axUz.set_xlabel(r"z/D")
    axUz.set_ylabel(r"$<w>/U_0$")
    axUz.set_xlim([-1.5, 1.5])
    axUz.set_ylim(uzYlim)
    axUz.legend(loc='upper right', frameon=False, bbox_to_anchor=(0.8, -0.15), ncol=2)
